# Integrated Industry Synthesis: A Governed Question Lifecycle for HKDSE ICT

**Project 7 · Applied Industry System Design, Ethical Reasoning, and Professional Defense**
MSc in AI Capstone · Draft v0.1 (Tasks 1-2 only)

*Working title:* Statistically grounded classification, generation, verification and adaptive delivery of exam-practice items under teacher governance.

| Task | Content | Status |
|---|---|---|
| 1 | Industry and problem space | Drafted below |
| 2 | Integration plan | Drafted below |
| 3 | System architecture and design | Not started |
| 4 | Construct the applied artifact | Not started |
| 5 | Evaluation | Not started |

**How the artifact is organised.** This notebook is the primary artifact. Reusable logic will live in importable modules under `src/` so that this notebook and an optional Streamlit front end call the same code. The front end (pending mentor approval, see Section 2.7) adds no AI method and contains no logic of its own.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Matplotlib  :", matplotlib.__version__)

---
# Task 1 - Industry and Problem Space

## 1.1 Industry and context

**Industry:** Education, specifically secondary-school examination preparation.
**Use case:** Practice for the Hong Kong Diploma of Secondary Education (HKDSE) Information and Communication Technology (ICT) Paper 1A, a 40-question multiple-choice paper.
**Data:** The 560-question bank of Paper 1A past papers (2012-2025) published by the Hong Kong Examinations and Assessment Authority (HKEAA) and used in Capstones 5 and 6. Each row has a question stem, four options, the official key, a curriculum classification (`qn_type_id`, modules A-E), a difficulty proxy (`cohort_pct`, the share of the cohort answering correctly), image references, and a cluster flag for questions that share a preamble or figure.

Reusing one domain and one bank across Capstones 5, 6 and 7 is deliberate. The brief asks for integration of *methods*, and a single coherent story is stronger evidence of intentional system design than three unrelated pieces.

## 1.2 The problem

My Capstone 6 tutor serves a finite bank of past-paper questions. Three gaps from the earlier capstones limit what it can do:

1. **The usable bank is small and uneven.** Capstone 6 used 519 questions (2012-2024). It excluded the whole 2025 paper (40 questions, none yet classified) and one withdrawn item, and it lumped 38 other unclassified questions into "general knowledge". Some sub-topics are so thin that the tutor runs out: in Capstone 6, sub-topic `d_cp_02` (3 questions) was exhausted on the fourth request and the tutor had no fallback.
2. **Generation exists but is not trustworthy.** In Capstone 5, a LoRA-tuned TinyLlama produced a valid multiple-choice format in 90% of samples (0% for the untuned base model), but only 1 of 10 samples was usable without editing. It produced no answer key, no distractor quality check and no difficulty estimate.
3. **Difficulty is only a proxy.** `cohort_pct` describes how past cohorts performed. It is not a calibrated difficulty estimate, and a newly generated item has no value for it at all.

## 1.3 Central question

> Can classified and generated practice items be added to an adaptive tutor's question bank safely, when every known weakness (missing labels, no answer key, weak distractors, unknown difficulty) is addressed by a measurable check and a human decision?

The aim is not a model that writes perfect questions. It is a *governed lifecycle* in which AI proposes, statistics and classifiers check, a teacher decides, and the tutor serves only what has been admitted and labels it honestly.

## 1.4 Why this problem suits an AI-based solution

- **Fixed supply, continuous demand.** Past papers are finite and cannot be re-issued, while students need targeted practice for specific topics.
- **Structured data.** Curriculum modules and sub-topics, cohort difficulty and cluster flags give each AI method something concrete to work on.
- **Each method addresses a different part of the problem.** Statistics measures and tests, classification labels, generation supplies, and an agentic tutor delivers and adapts. No single technique covers the whole problem, and Capstone 5 showed that generation alone is unsafe.

## 1.5 Constraints, risks and ethical considerations

| Area | Concern in this system | Planned response (detailed in Task 3) |
|---|---|---|
| Assessment integrity | A wrong key or a distractor that is also correct teaches a student something false, and the learner cannot tell | Independent blind-solve check, deterministic distractor checks, mandatory teacher approval |
| Difficulty calibration | A provisional estimate presented as measured would mislead students and teachers | Provisional flag on every generated or predicted item, shown wherever it is served |
| Intellectual property | HKEAA questions are copyright material, used here for non-commercial study only; a generator may memorise and reproduce them | Near-copy detection against the bank, no redistribution of the bank in the repository |
| Bias and equity | Thin sub-topics and the roughly one quarter of questions that depend on a figure are under-served; classifier or generator quality may differ by module | Per-module and per-year error analysis; abstention instead of forced labels |
| Minors' data | A deployed tutor would hold attempt records of secondary students | No real student data in this project; simulated sessions only; data minimisation and consent noted as deployment conditions (Hong Kong's Personal Data (Privacy) Ordinance, to be cited in the paper) |
| Accountability | A model cannot own the correctness of an exam item | A named teacher approves every item and every predicted label before it affects the tutor |
| Scope honesty | There are no real student outcomes, so learning gains cannot be claimed | Evaluation limited to item quality, gate behaviour, classifier accuracy and tool-level behaviour |

## 1.6 Scope

**In scope:** text-only multiple-choice items; module-level classification (sub-topic only where confidence is high); candidate generation and verification; teacher review; adaptive delivery of admitted items; statistical comparison of generated versus real items.

**Out of scope:** figure-aware generation or image interpretation; answer keys treated as authoritative; any deployment or learning-outcome claim; real student data.

## 1.7 Evidence from the question bank

The next cell quantifies the problem described above from the actual data, so the claims in Section 1.2 are checked rather than asserted. It needs `data/final_dataset.csv`.

> **Assumptions to confirm against `data/fields_explanation.md`:** column names (`qn_type_id`, `cohort_pct`, `correct_answer`, `qn_image_filename`, and a year column or a year encoded in `qn_id`), and that multiple labels in `qn_type_id` are separated by a comma, semicolon or pipe. The cell prints what it finds and fails softly if the file is missing.

In [ ]:
DATA_PATH = Path("data/final_dataset.csv")


def find_year(df):
    for c in ("paper_year", "year"):
        if c in df.columns:
            return pd.to_numeric(df[c], errors="coerce")
    # fall back to the first four-digit year found in qn_id
    return pd.to_numeric(df["qn_id"].astype(str).str.extract(r"(20\d{2})")[0], errors="coerce")


def split_labels(v):
    if pd.isna(v):
        return []
    return [t.strip() for t in str(v).replace(";", ",").replace("|", ",").split(",") if t.strip()]


if not DATA_PATH.exists():
    print(f"[skipped] {DATA_PATH} not found. Place the Kaggle file there and re-run this cell.")
    bank = None
else:
    bank = pd.read_csv(DATA_PATH)
    bank["year"] = find_year(bank)
    labels = bank["qn_type_id"].apply(split_labels)
    unlabelled = labels.apply(lambda L: (len(L) == 0) or all(t.lower() == "unclassified" for t in L))
    withdrawn = bank["correct_answer"].astype(str).str.strip() == "*"
    is_2025 = bank["year"] == 2025

    c6_usable = ~is_2025 & ~withdrawn
    n_total = len(bank)
    n_c6 = int(c6_usable.sum())
    n_2025 = int(is_2025.sum())
    n_unl_earlier = int((unlabelled & c6_usable).sum())
    n_potential = int((~withdrawn).sum())

    summary = pd.DataFrame({
        "count": [n_total, n_2025, int(withdrawn.sum()), n_c6, n_unl_earlier, n_potential],
    }, index=[
        "Total questions in file",
        "2025 questions (all unclassified)",
        "Withdrawn questions",
        "Capstone 6 usable bank (no 2025, no withdrawn)",
        "  of which unclassified (tagged general knowledge)",
        "Upper bound if 2025 is classified and approved",
    ])
    display(summary) if "display" in globals() else print(summary)

    if "qn_image_filename" in bank.columns:
        share_fig = bank["qn_image_filename"].notna().mean()
        print(f"\nQuestions that reference an image: {share_fig:.1%}")

    # sub-topic pool sizes in the Capstone 6 bank (multi-label: a question counts toward every tag)
    pools = (labels[c6_usable].explode().dropna()
             .loc[lambda s: s.str.lower() != "unclassified"].value_counts())
    print("\nSmallest sub-topic pools (Capstone 6 bank):")
    print(pools.tail(8).to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    pools.sort_values().plot.barh(ax=ax, color="#4a6fa5")
    ax.set_xlabel("Questions in bank (2012-2024)")
    ax.set_ylabel("Sub-topic tag")
    ax.set_title("Pool size per sub-topic: thin pools exhaust quickly")
    plt.tight_layout()
    plt.show()

---
# Task 2 - Integration Plan

## 2.1 Prior capstones integrated

| Capstone | Method family | What it contributes here | Why it is needed |
|---|---|---|---|
| **C6 Agentic AI** (orchestrator, 4 workers, 16 tools, least-privilege safeguards) | Agentic | The delivery layer: adaptive selection, grading, explanation, progress tracking, difficulty analysis. Also the *demand signal*: exhausted or thin pools | It is the system that consumes the bank and shows what is missing |
| **C5 Generative AI** (LoRA fine-tuned TinyLlama, structured evaluation) | Generative | The supply engine: candidate questions for a given module, sub-topic and difficulty band. Its documented failures define the requirements for the verification gate | Past papers are finite, and C5 showed generation helps but is unsafe on its own |
| **C3 Machine Learning** (supervised classification, tuning, feature importance, bias analysis) | Machine learning | (a) Classifies the unlabelled questions; (b) checks that generated items belong to the requested module; (c) per-module and per-year bias analysis | C5's TF-IDF module proxy was only 82% accurate on real questions; C6 had to exclude 2025 and lump 38 questions into "general knowledge" |
| **C2 Statistical Analysis** (cleaning, descriptive statistics, Welch's *t*-test, effect size) | Data and statistics | Gap detection; tests whether generated items differ from real ones (length, option structure, estimated difficulty); compares provisional with observed difficulty | Gives the system measurable, defensible checks rather than eyeballing |
| **C4 Deep Learning** (CNN image classifier) | Deep learning | **Not used.** About a quarter of questions depend on a figure, but figure-aware generation is out of scope, and a landmark CNN has no natural role | Stated explicitly so the exclusion is a design decision, not an omission |

C2 supplies *method* (hypothesis testing with effect sizes), not data: that project used the Prosper loan dataset, and the method is re-applied here to cohort difficulty and item properties.

## 2.2 How the components interact

The system runs as a loop. Each step depends on the output of the one before.

1. **Classify (C3).** Predict module labels for unlabelled questions (the 40 from 2025 and the 38 earlier unclassified ones). Evaluate on a *temporal* hold-out (train 2012-2022, test 2023-2024). Allow abstention below a confidence threshold. Predict at module level first; predict a sub-topic only above a high confidence. Keep the multi-label convention from C6. Store predictions in separate columns (`predicted_qn_type`, `label_source`, `confidence`); never overwrite official labels or the existing "general knowledge" tag with a guess.
2. **Teacher approves labels.** Predicted labels reach the tutor only after approval. Only then can the bank grow, from 519 up to a maximum of 559 questions.
3. **Detect gaps (C6 Analyst + C2).** Find (module, sub-topic, difficulty band) cells where the pool is thin or was exhausted in a session.
4. **Generate candidates (C5).** Request candidates for those cells. The generator is backend-agnostic and driven by a config entry per model, so a bake-off costs little extra code (Section 2.4, decision 5).
5. **Verify (gate).** Deterministic checks (format, duplicate or overlapping options, near-copy of real questions, key much longer than distractors); an independent blind solver from a *different model family* than the generator; the C3 classifier confirming module fit; and C2 tests comparing candidate properties with real items (Welch's *t*-test, Cohen's *d*).
6. **Teacher reviews.** A teacher supplies or confirms the key and approves, edits or rejects. Rejection reasons are logged.
7. **Admit as provisional.** Approved items enter the bank flagged `provisional=True` with an estimated difficulty marked as low confidence.
8. **Deliver (C6).** The tutor serves provisional items only when the real pool for that cell is exhausted, and always labels them as provisional.
9. **Feedback (C2 + C6 Tracker).** Observed correct rates are compared with the provisional estimate; large mismatches flag an item for recalibration or retirement. With no real students, this loop is demonstrated on simulated attempts and labelled as such.

```mermaid
flowchart TD
  U[Unlabelled questions<br/>2025 + 38 earlier] --> C[C3 classifier<br/>temporal split, abstention]
  C --> TL[Teacher approves labels]
  TL --> B[(Question bank<br/>519 up to 559)]
  B --> A[C6 Analyst + C2 stats<br/>gap detection]
  A --> G[C5 generator<br/>config-driven backends]
  G --> V[Gate<br/>deterministic checks, blind solver,<br/>C3 module check, C2 tests]
  V -- fail --> R[Rejection log]
  V -- pass --> T[Teacher review<br/>key + approve / edit / reject]
  T -- reject --> R
  T -- approve --> P[Provisional items]
  P --> B
  B --> S[C6 tutor<br/>serve, grade, explain]
  S --> K[C6 Tracker<br/>attempts]
  K --> F[C2 provisional vs observed difficulty]
  F --> A
```

*A rendered architecture diagram, in the same style as Capstone 6's `architecture_diagram.png`, will be produced in Task 3.*

## 2.3 What makes this integration rather than a collection

- **Each earlier limitation becomes a requirement here.** Thin pools drive generation, the 1-in-10 usable rate drives the gate, the weak module proxy drives the classifier, and the difficulty proxy drives calibration.
- **One shared bank and one schema.** Every component reads and writes the same bank, including the v1.6 `cluster` fields, and the loop closes through the statistics.
- **It is new work.** The classification pipeline for unlabelled items, the gate, the teacher review queue, the provisional flag, the generator bake-off and the feedback loop do not exist in any earlier capstone. Earlier code is reused as modules, not resubmitted.

## 2.4 Design decisions made so far

| # | Decision | Reason |
|---|---|---|
| 1 | Same domain and bank as C5 and C6 | One coherent story; the brief asks for integration of methods, not datasets |
| 2 | Integrate C2, C3, C5, C6; do not use C4 | Each of the four has a real role; C4 has none, and this is stated openly |
| 3 | C3 classifies the 2025 and other unlabelled questions, and also checks generated items | Grows the usable bank and gives the classifier two jobs |
| 4 | Temporal split, module-level first, abstention, multi-label kept, predictions stored separately and teacher-approved | 2025 has no ground truth, so honest evaluation needs a held-out year range; wrong labels would corrupt C6 mastery updates |
| 5 | The generator is backend-agnostic, with one config entry per model. Compare the Capstone 5 TinyLlama models, newer open models (for example Phi-4-mini, Gemma 4, Qwen3) and Claude Haiku on identical targets | Tests whether the C5 limitation was a model-size problem or a method problem. Model identifiers must be verified before running |
| 6 | Prefer grounded prompting (curriculum text, same-sub-topic real examples, structured JSON output with a per-option rationale) over further fine-tuning or GRPO | 445 training examples are too few; fine-tuning is not the bottleneck. GRPO with the gate as reward is a future extension |
| 7 | The gate combines deterministic checks, an independent blind solver from a different model family, the C3 module check and C2 distribution tests | Self-agreement between same-family models is not evidence of correctness |
| 8 | A teacher approves every item and label; a generated key is never authoritative; generated items are always provisional | Accountability stays with a human; the system claims to *reduce review effort*, not to produce correct questions |
| 9 | Notebook is the primary artifact, with logic in `src/` modules; a Streamlit chat front end is a thin layer over the same modules; cached outputs provide an offline mode | Reproducibility and rubric fit come first; the front end makes the chat, review queue and routing visible in the defence |
| 10 | Text-only items; no figure-aware generation | About a quarter of real questions depend on a figure; out of scope |
| 11 | No real student data and no learning-outcome claims | Minors' data protection and scope honesty |

## 2.5 Boundaries of capability and responsibility

The system **proposes, checks, flags and delivers**. It never decides alone that an item is correct, never serves an unreviewed item, never overwrites an official label, and never claims that students learn better. The teacher owns correctness; the tutor owns only delivery and feedback on admitted items.

## 2.6 Preview of the evaluation (Task 5)

All of it is possible without real students:

- **Classifier:** accuracy, macro-F1 and abstention rate on the 2023-2024 hold-out against a TF-IDF baseline like C5's proxy; per-module and per-year error table; separate reporting of 2025 confidence.
- **Generator bake-off:** the same targets through each backend, scored blind on a short rubric (format, single correct answer, distractor quality, curriculum fit, level, originality); usable-without-edits rate with confidence intervals, compared with C5's 1-in-10 baseline.
- **Gate:** how often it rejects items the teacher also rejects, and how often it wrongly rejects acceptable items.
- **Statistics:** Welch's *t*-test and effect size for generated versus real items on measurable properties; calibration of the provisional difficulty estimate on held-out real questions.
- **Tutor behaviour:** deterministic scenarios (thin pool, provisional labelling, a rejected item never served) in the style of C6's tool-level demonstrations.

## 2.7 Repository plan and reuse of prior work

| New module | Role | Comes from |
|---|---|---|
| `src/data.py` | Load bank, apply exclusions, derive clusters | C6 data loading |
| `src/stats.py` | Gap detection, Welch's *t*-test, effect size | C2 methods |
| `src/classifier.py` | Module classifier, abstention, bias table | C3 workflow |
| `src/generator.py` | Backend-agnostic candidate generation | C5 model and prompt design |
| `src/gate.py`, `src/review.py` | Verification checks, review queue, provisional flag | New |
| `src/tutor/` | Tools, agents, orchestrator | C6, minimal changes |
| `app/streamlit_app.py` | Presentation layer, no logic | New (needs mentor approval) |

## 2.8 Open items

- [ ] **Mentor approval for Streamlit.** The brief says no new technologies outside coursework. If declined, fall back to notebook transcripts and ipywidgets-style views.
- [ ] **Verify dataset column names** and the multi-label delimiter against `fields_explanation.md`.
- [ ] **Confirm the C5 LoRA adapter and conditioning header are saved**, so the C5 baselines can be reproduced in the bake-off.
- [ ] **Verify model identifiers** for the open models before the bake-off, and decide whether any needs 4-bit loading (an addition outside the coursework libraries).
- [ ] **Decide the held-out and approval protocol** for the 2025 labels (who approves, and on what sample).
- [ ] **Citations for the paper:** classification and bias evaluation, LLM-as-solver and self-consistency limits, assessment-item quality, and responsible-AI governance in education. The notebook does not need citations; the paper does.